## Data Import 
Importing small sample of data to use as a prototype. Here, we will be using python with the pandas library to attain a performance baseline.

In [1]:
import pandas as pd

df = pd.read_csv("datasets/cyclistic_tripdata_2021.csv", nrows=10000)
df.head()

,ride_id,rideable_type,started_at,ended_at,start_station_name,start_station_id,end_station_name,end_station_id,start_lat,start_lng,end_lat,end_lng,member_casual
0,31FAE55254BEE127,classic_bike,2021-02-12 09:40:15 UTC,2021-02-12 13:28:31 UTC,Fairbanks Ct & Grand Ave,TA1305000003,NaN,NaN,41.891847,-87.620580,NaN,NaN,casual
1,ED658F5C9D645C49,classic_bike,2021-02-14 18:41:23 UTC,2021-02-15 14:08:36 UTC,Clark St & Wrightwood Ave,TA1305000014,NaN,NaN,41.929546,-87.643118,NaN,NaN,member
2,AB91DE991455DBCB,classic_bike,2021-02-14 17:45:13 UTC,2021-02-15 18:45:08 UTC,Clark St & Lake St,KA1503000012,NaN,NaN,41.886021,-87.630876,NaN,NaN,member
3,6E37DC248B3CFE21,classic_bike,2021-02-20 14:34:21 UTC,2021-02-21 15:34:15 UTC,Columbus Dr & Randolph St,13263,NaN,NaN,41.884728,-87.619521,NaN,NaN,member
4,985662404D86B374,classic_bike,2021-02-27 13:22:20 UTC,2021-02-28 14:22:14 UTC,Clark St & Drummond Pl,TA1307000142,NaN,NaN,41.931248,-87.644336,NaN,NaN,casual


Calculate ride duration, allows us to perform better analysis later on

In [2]:
df["started_at"] = pd.to_datetime(df["started_at"], utc=True)
df["ended_at"] = pd.to_datetime(df["ended_at"], utc=True)


In [3]:
df["ride_length"] = df["ended_at"] - df["started_at"]
df["ride_length_minutes"] = df["ride_length"].dt.total_seconds() / 60
df.head()


,ride_id,rideable_type,started_at,ended_at,start_station_name,start_station_id,end_station_name,end_station_id,start_lat,start_lng,end_lat,end_lng,member_casual,ride_length,ride_length_minutes
0,31FAE55254BEE127,classic_bike,2021-02-12 09:40:15+00:00,2021-02-12 13:28:31+00:00,Fairbanks Ct & Grand Ave,TA1305000003,NaN,NaN,41.891847,-87.620580,NaN,NaN,casual,0 days 03:48:16,228.266667
1,ED658F5C9D645C49,classic_bike,2021-02-14 18:41:23+00:00,2021-02-15 14:08:36+00:00,Clark St & Wrightwood Ave,TA1305000014,NaN,NaN,41.929546,-87.643118,NaN,NaN,member,0 days 19:27:13,1167.216667
2,AB91DE991455DBCB,classic_bike,2021-02-14 17:45:13+00:00,2021-02-15 18:45:08+00:00,Clark St & Lake St,KA1503000012,NaN,NaN,41.886021,-87.630876,NaN,NaN,member,1 days 00:59:55,1499.916667
3,6E37DC248B3CFE21,classic_bike,2021-02-20 14:34:21+00:00,2021-02-21 15:34:15+00:00,Columbus Dr & Randolph St,13263,NaN,NaN,41.884728,-87.619521,NaN,NaN,member,1 days 00:59:54,1499.900000
4,985662404D86B374,classic_bike,2021-02-27 13:22:20+00:00,2021-02-28 14:22:14+00:00,Clark St & Drummond Pl,TA1307000142,NaN,NaN,41.931248,-87.644336,NaN,NaN,casual,1 days 00:59:54,1499.900000


## Quality checks

Looking for missing values
Filter out any negative or >24 hr rides

Now we can see the size difference between the clean and unclean datasets, it's only 11 but it is 11 values that could have seriously affected the results of our analysis.

In [4]:
missing_values = df.isna().sum().sort_values(ascending=False)
print("Missing values per column:\n", missing_values)

print("\nRide length (minutes) summary:")
print(df["ride_length_minutes"].describe())

max_duration_minutes = 24 * 60
mask_valid_duration = (df["ride_length_minutes"] > 0) & (df["ride_length_minutes"] <= max_duration_minutes)

print("\nNumber of rides before filtering:", len(df))
print("Number of rides after filtering:", mask_valid_duration.sum())

df_clean = df[mask_valid_duration].copy()

Missing values per column:
 end_station_name       1033
end_station_id         1033
start_station_name      876
start_station_id        876
end_lat                  16
end_lng                  16
started_at                0
ended_at                  0
ride_id                   0
rideable_type             0
start_lat                 0
start_lng                 0
member_casual             0
ride_length               0
ride_length_minutes       0
dtype: int64

Ride length (minutes) summary:
count    10000.000000
mean        20.634535
std        307.618286
min          0.000000
25%          5.933333
50%          9.850000
75%         17.516667
max      30129.233333
Name: ride_length_minutes, dtype: float64

Number of rides before filtering: 10000
Number of rides after filtering: 9989


## Member vs casual
Compare ride lengths and trip counts between members and casual riders. We can see that the bikes are more often used by members on short journeys (like going to work) and sometimes used by casuals for longer journeys (likely tourists viewing the city)

In [5]:
ride_length_summary = df_clean.groupby("member_casual")["ride_length_minutes"].describe()
print("Ride length (minutes) by user type:\n", ride_length_summary)

trip_counts = df_clean["member_casual"].value_counts().rename("trip_count")
trip_share = df_clean["member_casual"].value_counts(normalize=True).rename("share")

print("\nTrip share by user type:\n", (trip_share * 100).round(2).astype(str) + "%")

Ride length (minutes) by user type:
                 count       mean        std       min    25%        50%  \
member_casual                                                             
casual         1986.0  26.531898  74.601110  0.116667  7.800  13.925000   
member         8003.0  13.716102  30.930516  0.016667  5.625   9.166667   

                   75%          max  
member_casual                        
casual         25.6625  1427.633333  
member         15.8000  1308.100000  

Trip share by user type:
 member_casual
member    80.12%
casual    19.88%
Name: share, dtype: str


## Making Time and Day cols
Allows for us to perform day and time analysis

In [6]:
df_clean["hour"] = df_clean["started_at"].dt.hour
df_clean["day_of_week"] = df_clean["started_at"].dt.day_name()

df_clean[["started_at", "hour", "day_of_week"]].head()

,started_at,hour,day_of_week
0,2021-02-12 09:40:15+00:00,9,Friday
1,2021-02-14 18:41:23+00:00,18,Sunday
5,2021-02-17 16:21:59+00:00,16,Wednesday
6,2021-02-06 06:31:54+00:00,6,Saturday
8,2021-02-18 17:17:23+00:00,17,Thursday


## Time and Day patterns

Examine how rides vary by hour of day and day of week for members vs casual riders.

It's pretty clear to see that members are commuting to and from work (peak at 5pm, the time most people get off work), it looks like casual users are more likely tourists just staking leisurely rides around the city (peak at 2pm, while most people would usually be in work).

In [7]:
hourly_counts = df_clean.groupby(["hour", "member_casual"]).size().unstack(fill_value=0)

print("Trips by hour of day and user type (first few rows):")
print(hourly_counts.head)

print("\nSummary statistics by hour:")
print(hourly_counts.describe())

print("\nPeak hour by user type:")
for user_type in hourly_counts.columns:
    peak_hour = hourly_counts[user_type].idxmax()
    peak_value = hourly_counts[user_type].max()
    print(f"- {user_type}: peak at hour {peak_hour} with {peak_value} trips")

day_order = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]

weekday_counts = (
    df_clean.groupby(["day_of_week", "member_casual"])
    .size()
    .unstack(fill_value=0)
    .reindex(day_order)
)

Trips by hour of day and user type (first few rows):
<bound method NDFrame.head of member_casual  casual  member
hour                         
0                  29      41
1                  14      18
2                  12      16
3                  12      15
4                   8      13
5                  18      96
6                  22     256
7                  48     430
8                  62     435
9                  58     336
10                 87     364
11                137     497
12                176     557
13                148     591
14                194     606
15                185     655
16                185     707
17                165     786
18                139     632
19                 79     385
20                 66     256
21                 59     123
22                 51     113
23                 32      75>

Summary statistics by hour:
member_casual      casual      member
count           24.000000   24.000000
mean            82.750000  333.

## Analysing Daily Trends
we can see saturday is extra busy for casual users which supports the theory they are tourists trying to view the city

In [8]:
member_counts = weekday_counts["member"]
casual_counts = weekday_counts["casual"]


print("Trips by day of week and user type (counts):")
print(weekday_counts)

print("\nDay-of-week share of trips by user type (row %):")
weekday_pct = weekday_counts.div(weekday_counts.sum(axis=1), axis=0).round(3)
print(weekday_pct)

print("\nKey weekday patterns:")

for day in weekday_counts.index:
    casual = weekday_counts.loc[day, "casual"]
    member = weekday_counts.loc[day, "member"]
    print(f"- {day}: {casual} casual vs {member} member trips")

Trips by day of week and user type (counts):
member_casual  casual  member
day_of_week                  
Monday            201    1000
Tuesday           197    1125
Wednesday         216    1248
Thursday          240    1161
Friday            287    1293
Saturday          528    1312
Sunday            317     864

Day-of-week share of trips by user type (row %):
member_casual  casual  member
day_of_week                  
Monday          0.167   0.833
Tuesday         0.149   0.851
Wednesday       0.148   0.852
Thursday        0.171   0.829
Friday          0.182   0.818
Saturday        0.287   0.713
Sunday          0.268   0.732

Key weekday patterns:
- Monday: 201 casual vs 1000 member trips
- Tuesday: 197 casual vs 1125 member trips
- Wednesday: 216 casual vs 1248 member trips
- Thursday: 240 casual vs 1161 member trips
- Friday: 287 casual vs 1293 member trips
- Saturday: 528 casual vs 1312 member trips
- Sunday: 317 casual vs 864 member trips


## Ride length distributions
We've capped this at 2 hour rides to ignore some outliers as to not disrupt our analysis. 
We can see that members favour shorter rides, since we believe they're commuting to work, perhaps it is inner city residents who avail of these bikes most, as the data would suggest. Potentially people cycle from work to bus/train stations to travel on farther from.
Casual users favour slightly longer rides, the presence of shorter rides in the most common ride lenghts does still uphold our theory of casuals representing tourists, since tourists most likely would not have a car to commute around with, bikes are a cheap and easy option.

In [9]:
max_minutes = 120  #this is capped at 2 hours to ignore outliers, it only cuts out ~60 that seriously alter the mean etc, it also shows a cleaner version of data we saw earlier
subset = df_clean[df_clean["ride_length_minutes"] <= max_minutes].copy()

print("Ride length minutes summary by user type (<= 120 mins):")
summary = (
    subset.groupby("member_casual")["ride_length_minutes"]
    .describe(percentiles=[0.25, 0.5, 0.75])
    .round(1)
)

print(summary)

print("\nTop 10 most common ride lengths (rounded to nearest minute) by user type:")

subset["ride_length_round"] = subset["ride_length_minutes"].round()
mode_table = (
    subset.groupby(["member_casual", "ride_length_round"])
    .size()
    .reset_index(name="count")
    .sort_values(["member_casual", "count"], ascending=[True, False])
)


for user_type in subset["member_casual"].unique():
    print(f"\nUser type: {user_type}")
    print(mode_table[mode_table["member_casual"] == user_type].head(10))

Ride length minutes summary by user type (<= 120 mins):
                count  mean   std  min  25%   50%   75%    max
member_casual                                                 
casual         1945.0  19.5  18.3  0.1  7.7  13.6  24.6  118.0
member         7974.0  12.4  10.6  0.0  5.6   9.1  15.7  113.2

Top 10 most common ride lengths (rounded to nearest minute) by user type:

User type: member
    member_casual  ride_length_round  count
107        member                6.0    641
108        member                7.0    600
106        member                5.0    596
109        member                8.0    537
105        member                4.0    524
110        member                9.0    442
111        member               10.0    411
104        member                3.0    405
112        member               11.0    365
113        member               12.0    305

User type: casual
   member_casual  ride_length_round  count
8         casual                8.0    104
7        

## Bike type usage
Here, we wanted to find the relationship between bike type and user type. We found the average ride duration for each bike and user combination. This analysis helps identify user preferences for specific bike types and highlights the differences between members and casual riders that we have previously theorised about. 
The shear number of member usage is higher, but when we look at the percentage usage, the figures start to make sense. Members seem to prefer the "classic bike" over the electric more so than the casual users do, this could be down to members being into fitness or simply the bike availability in their residential areas.
The only question we are left with right now is why do only casual users use "docked bikes"?

In [10]:
bike_user_ct = pd.crosstab(df_clean["rideable_type"], df_clean["member_casual"])
print("Bike type x user type (counts):")
print(bike_user_ct)


print("\nBike type x user type (row %):")
bike_pct = bike_user_ct.div(bike_user_ct.sum(axis=1), axis=0).round(3)
print(bike_pct)

print("\nAverage ride length (minutes) by bike type and user type:")

avg_duration_bike_user = (
    df_clean.groupby(["rideable_type", "member_casual"])["ride_length_minutes"]
    .mean()
    .round(1)
    .unstack()
)
print(avg_duration_bike_user)

print("\nInterpretation by bike type:")
for bike in bike_user_ct.index:
    casual_share = bike_pct.loc[bike, "casual"] * 100
    member_share = bike_pct.loc[bike, "member"] * 100
    print(f"- {bike}: {casual_share:.1f}% casual, {member_share:.1f}% member")

Bike type x user type (counts):
member_casual  casual  member
rideable_type                
classic_bike     1033    5842
docked_bike       249       0
electric_bike     704    2161

Bike type x user type (row %):
member_casual  casual  member
rideable_type                
classic_bike    0.150   0.850
docked_bike     1.000   0.000
electric_bike   0.246   0.754

Average ride length (minutes) by bike type and user type:
member_casual  casual  member
rideable_type                
classic_bike     26.9    14.1
docked_bike      56.3     NaN
electric_bike    15.5    12.6

Interpretation by bike type:
- classic_bike: 15.0% casual, 85.0% member
- docked_bike: 100.0% casual, 0.0% member
- electric_bike: 24.6% casual, 75.4% member


## Station-based insights
We wanted to find out which stations had the highest numbers of users arriving and departing from them, we can see that "Dearborn St and Erie St" appears one of the focal points, seeing as it is the most popular destination and second most popular starting point. Similar observations can be made for each other station.
We also computed the average ride duration by start station an user type. We filtered this to include stations with 30+ trips to ensure reliability.
Using this information, we can detarmine what areas are residential and which are tourist hotspots among many other things.

In [11]:
for user_type in ["member", "casual"]:
    print(f"\nTop 10 start stations for {user_type}s:")
    top_start = (
        df_clean[df_clean["member_casual"] == user_type]["start_station_name"]
        .value_counts()
        .head(10)
    )
    print(top_start)

    print(f"\nTop 10 end stations for {user_type}s:")
    top_end = (
        df_clean[df_clean["member_casual"] == user_type]["end_station_name"]
        .value_counts()
        .head(10)
    )
    print(top_end)

min_trips_per_station = 30

avg_duration_station = (
    df_clean.groupby(["start_station_name", "member_casual"])["ride_length_minutes"]
    .agg(["count", "mean"])
    .reset_index()
)



avg_duration_station = avg_duration_station[avg_duration_station["count"] >= min_trips_per_station]

print("\nAverage ride length (minutes) by start station and user type (>= 30 trips):")
print(avg_duration_station.sort_values("mean", ascending=False).head(20))


Top 10 start stations for members:
start_station_name
Dearborn St & Erie St        82
Clark St & Elm St            78
Wells St & Elm St            78
Wells St & Huron St          74
Desplaines St & Kinzie St    68
Kingsbury St & Kinzie St     67
Clinton St & Madison St      66
Daley Center Plaza           64
St. Clair St & Erie St       64
Columbus Dr & Randolph St    64
Name: count, dtype: int64

Top 10 end stations for members:
end_station_name
Dearborn St & Erie St        102
Clark St & Elm St             93
St. Clair St & Erie St        76
Wabash Ave & Roosevelt Rd     73
Wells St & Elm St             65
Kingsbury St & Kinzie St      63
Clinton St & Madison St       62
Broadway & Barry Ave          59
McClurg Ct & Erie St          58
Wells St & Concord Ln         56
Name: count, dtype: int64

Top 10 start stations for casuals:
start_station_name
Lake Shore Dr & Monroe St    29
Ellis Ave & 60th St          25
Millennium Park              25
Streeter Dr & Grand Ave      19
Daley Cen

## Time of day x day of week bike usage
We could use seaborn to display this information on a heatmap as it is very suited for that, but seeing as we are attempting to set a base line for time, we thought it best to stick with text outputs in case graphics take a significantly long amount of time to display.
Now bike usage by day of week and time is legible. The huge numbers of users visible during the work rush hours on weekdays confirms our earlier hypotheses. The low numbers of rides over the weekend do so as well, likely being mostly tourist use.

In [14]:
#this is basically a text heat map, I wanted to make something easily duplicated in other languages without importing libraries

cat_type = pd.CategoricalDtype(categories=day_order, ordered=True)
df_clean["day_of_week"] = df_clean["day_of_week"].astype(cat_type)

for user_type in ["member", "casual"]:
    subset = df_clean[df_clean["member_casual"] == user_type]
    pivot = subset.pivot_table(
        index="day_of_week",
        columns="hour",
        values="ride_id",
        aggfunc="count",
        fill_value=0,
    )

    print(f"\nTrips table (day x hour) for {user_type}:")
    print(pivot)

    print(f"\nTop 5 day-hour cells for {user_type}:")
    top_cells = (
        pivot.stack()
        .sort_values(ascending=False)
        .head(5)
        .reset_index(name="trips")
    )
    for _, row in top_cells.iterrows():
        print(f"- {row['day_of_week']} @ {row['hour']}: {row['trips']} trips")


Trips table (day x hour) for member:
hour         0   1   2   3   4   5   6   7   8   9   ...   14   15   16   17  \
day_of_week                                          ...                       
Monday        3   1   4   1   1  15  48  65  53  38  ...   85   66   97   94   
Tuesday       3   1   0   1   1  18  42  80  56  49  ...   64   93  112  130   
Wednesday     2   0   2   2   3  18  52  87  81  39  ...   79  101  126  116   
Thursday      4   2   0   4   3  15  44  82  84  56  ...   67   71  108  140   
Friday        6   3   1   2   3  24  48  73  92  64  ...   74  108  101  139   
Saturday      9   5   6   2   1   3  16  27  44  60  ...  142  125   99  106   
Sunday       14   6   3   3   1   3   6  16  25  30  ...   95   91   64   61   

hour          18  19  20  21  22  23  
day_of_week                           
Monday        95  57  31  12  14  10  
Tuesday       94  59  34  16  18   6  
Wednesday    107  70  48  17  13   6  
Thursday     109  51  36  22   9  12  
Friday 

## Rush hour commuter usage
To make the above information more legible, we have targeted three main points:

Members take the majority of trips overall, particularly on weekdays (5,827 vs 2,176 weekend trips), suggesting strong commuter usage. Casual riders show a relatively higher weekend share (845 vs 1,141 weekday trips), indicating the touristic tendancies we have previously mentioned..

Commute hour trips are almost evenly split for members (4,036 non-commute vs 3,967 commute), reinforcing their commuter profile, while casual riders ride more outside commute hours.

In terms of duration, 73.0% of member trips are short ( less than 15 minutes), compared to 53.2% for casual riders. Casual users have a much higher proportion of long trips (> 45 minutes: 10.3% vs 1.9%), further supporting the tourist vs commuter distinction.

In [13]:
df_clean["is_weekend"] = df_clean["day_of_week"].isin(["Saturday", "Sunday"])

commute_hours = list(range(6, 10)) + list(range(16, 20))
df_clean["is_commute_hour"] = df_clean["hour"].isin(commute_hours)


bins = [0, 15, 45, float("inf")]
labels = ["short (≤15m)", "medium (15–45m)", "long (>45m)"]
df_clean["duration_band"] = pd.cut(df_clean["ride_length_minutes"], bins=bins, labels=labels, right=True)


weekend_summary = (
    df_clean
    .groupby(["member_casual", "is_weekend"])["ride_id"]
    .count()
    .unstack(fill_value=0)
)
print("Weekend vs weekday trips by user type:\n", weekend_summary)

commute_summary = (
    df_clean
    .groupby(["member_casual", "is_commute_hour"])["ride_id"]
    .count()
    .unstack(fill_value=0)
)
print("\nCommute-hour vs other-hour trips by user type:\n", commute_summary)


band_summary = pd.crosstab(df_clean["member_casual"], df_clean["duration_band"], normalize="index") * 100
print("\nRide length bands (% of trips) by user type:\n", band_summary.round(1))

Weekend vs weekday trips by user type:
 is_weekend     False  True 
member_casual              
casual          1141    845
member          5827   2176

Commute-hour vs other-hour trips by user type:
 is_commute_hour  False  True 
member_casual                
casual            1228    758
member            4036   3967

Ride length bands (% of trips) by user type:
 duration_band  short (≤15m)  medium (15–45m)  long (>45m)
member_casual                                            
casual                 53.2             36.5         10.3
member                 73.0             25.1          1.9
